In [14]:
import warnings
import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

In [15]:
df = pd.read_csv("final_preprocessed_movies.csv")
df

,id,title,overview,poster_path,release_date,release_year,vote_average,vote_count,runtime,original_language,popularity,genres,budget,keywords,content,revenue,original_title
0,27205,Inception,"Cobb, a skilled thief who commits corporate es...",/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,2010-07-15,2010,8.364,34495,148,en,83.952,"Action, Science Fiction, Adventure",160000000,"rescue, mission, dream, airplane, paris, franc...","Cobb, a skilled thief who commits corporate es...",825532764,Inception
1,157336,Interstellar,The adventures of a group of explorers who mak...,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,2014-11-05,2014,8.417,32571,169,en,140.241,"Adventure, Drama, Science Fiction",165000000,"rescue, future, spacecraft, race against time,...",The adventures of a group of explorers who mak...,701729206,Interstellar
2,155,The Dark Knight,Batman raises the stakes in his war on crime. ...,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,2008-07-16,2008,8.512,30619,152,en,130.643,"Drama, Action, Crime, Thriller",185000000,"joker, sadism, chaos, secret identity, crime f...",Batman raises the stakes in his war on crime. ...,1004558444,The Dark Knight
3,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...",/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,2009-12-15,2009,7.573,29815,162,en,79.932,"Action, Adventure, Fantasy, Science Fiction",237000000,"future, society, culture clash, space travel, ...","In the 22nd century, a paraplegic Marine is di...",2923706026,Avatar
4,24428,The Avengers,When an unexpected enemy emerges and threatens...,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,2012-04-25,2012,7.710,29166,143,en,98.082,"Science Fiction, Action, Adventure",220000000,"new york city, superhero, shield, based on com...",When an unexpected enemy emerges and threatens...,1518815515,The Avengers
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27837,341886,June,June is different from all other nine-year old...,/zvHlijzVA5hNOmy6vMi0vSlnlsh.jpg,2015-08-01,2015,3.900,50,84,en,3.207,"Science Fiction, Horror",0,"exorcism, possession",June is different from all other nine-year old...,0,June
27838,13021,Flakes,Aspiring rock musician Neal Downs manages a ce...,/2e5t3EUL0gwYs78Ah4XFGpfmJZl.jpg,2007-12-19,2007,5.200,50,84,en,3.707,"Comedy, Romance",0,NaN,Aspiring rock musician Neal Downs manages a ce...,0,Flakes
27839,53411,Shanghaied,A shipowner intends to scuttle his ship on its...,/juwx4pw6gyloHyxAEUagCF3vkB6.jpg,1915-10-04,1915,5.830,50,27,en,3.418,Comedy,0,"black and white, silent film, short film",A shipowner intends to scuttle his ship on its...,0,Shanghaied
27840,144651,Ghosts Before Breakfast,"Hans Richter, noted for his abstract shorts, h...",NaN,1928-07-14,1928,6.900,50,9,de,2.173,"Animation, Comedy",0,"surrealism, avant-garde, dadaism","Hans Richter, noted for his abstract shorts, h...",0,Vormittagsspuk


In [16]:
df[df['title'] == 'Titanic']

,id,title,overview,poster_path,release_date,release_year,vote_average,vote_count,runtime,original_language,popularity,genres,budget,keywords,content,revenue,original_title
17,597,Titanic,101-year-old Rose DeWitt Bukater tells the sto...,/9xjZS2rlVxm8SFx8kPC3aIGCOYQ.jpg,1997-11-18,1997,7.900,23637,194,en,102.348,"Drama, Romance",200000000,"epic, ship, drowning, panic, shipwreck, evacua...",101-year-old Rose DeWitt Bukater tells the sto...,2264162353,Titanic
16183,16535,Titanic,"Unhappily married, Julia Sturges decides to go...",/rEPzO9I6LCk6Mxg1X4BsBk6oA3V.jpg,1953-04-11,1953,6.600,120,98,en,17.586,"Drama, Romance",1805000,titanic,"Unhappily married, Julia Sturges decides to go...",4905000,Titanic
23391,11021,Titanic,This little-known German film retells the true...,/Al7oIXQ4dZAofBTZWm6OiXS3MEa.jpg,1943-11-10,1943,6.174,66,85,de,14.382,"Action, Drama, History",0,"sea, captain, passenger, cruise, iceberg, tita...",This little-known German film retells the true...,0,Titanic


In [20]:
num = [
    "vote_average", "vote_count", "runtime",
    "popularity", "release_year", "budget"
]

genres = df["genres"].str.get_dummies(sep=", ")

languages = pd.get_dummies(
    df["original_language"].where(
        df["original_language"].isin(
            df["original_language"].value_counts().head(10).index
        ),
        "other"
    ) 
)

tfidf = TfidfVectorizer(max_features=75, stop_words="english")
content_features = tfidf.fit_transform(df["content"]).toarray()

In [21]:
num = StandardScaler().fit_transform(df[num])

In [22]:
X = np.concatenate([
    num,
    genres.values,
    languages.values,
    content_features
], axis=1)

print("Feature Matrix:", X.shape)

Feature Matrix: (27842, 111)


In [23]:
model_k = KMeans(
    n_clusters=14,
    random_state=42,
    n_init=10
)

df["cluster"] = model_k.fit_predict(X)

In [24]:
print("WCSS:", round(model_k.inertia_, 2))
print("Silhouette:", round(
    silhouette_score(X, df["cluster"]), 4
))

WCSS: 128188.22
Silhouette: 0.0931


In [25]:
model_bundle = {"model": model_k, "df": df, "X": X, "numeric_columns": num, "genre_columns": genres.columns.tolist(), "language_columns": languages.columns.tolist(), "tfidf": tfidf}

In [26]:
with open('clustering_model.pkl','wb') as file:
    pickle.dump(model_bundle,file)

In [27]:
with open('clustering_model.pkl','rb') as file:
    model_bundle = pickle.load(file)

In [28]:
def recommend_movie(title, n=10):
    model = model_bundle["model"]
    df = model_bundle["df"]
    X = model_bundle["X"]

    movie = df[df["title"].astype(str).str.lower() == title.strip().lower()]
    if movie.empty:
        return None
    
    movie_index = movie.index[0]
    movie_features = X[movie_index].reshape(1, -1)
    prediction = model.predict(movie_features)
    cluster = int(prediction[0])
    cluster_indices = df[df["cluster"] == cluster].index

    distances = euclidean_distances(movie_features, X[cluster_indices])[0]
    result = pd.DataFrame({"index": cluster_indices, "distance": distances})
    result = result[result["index"] != movie_index]
    result = result.sort_values("distance")
    result = result.head(n)

    cols = [
        "id", "title", "original_title", "overview", "poster_path",
        "release_date", "release_year", "vote_average", "vote_count",
        "runtime", "genres", "original_language", "popularity", "budget", "revenue"
    ]
    available_cols = [c for c in cols if c in df.columns]
    rec_df = df.loc[result["index"], available_cols].copy()
    rec_df["similarity"] = [int(round(max(60, min(99, 100 - (d * 5))))) for d in result["distance"]]
    if "poster_path" in rec_df.columns:
        rec_df["poster_path"] = rec_df["poster_path"].apply(
            lambda x: f"https://image.tmdb.org/t/p/w500{x}" if isinstance(x, str) and x.startswith("/") else (x if isinstance(x, str) else "")
        )
    recommendations = rec_df.fillna("").to_dict(orient="records")

    queried_movie = df.loc[movie_index, available_cols].to_dict()
    if isinstance(queried_movie.get("poster_path"), str) and queried_movie["poster_path"].startswith("/"):
        queried_movie["poster_path"] = f"https://image.tmdb.org/t/p/w500{queried_movie['poster_path']}"

    return {
        "movie": queried_movie,
        "cluster": cluster,
        "recommendations": recommendations
    }


In [30]:
recommend_movie('Dog Day Afternoon',10)

{'movie': {'id': 968,
  'title': 'Dog Day Afternoon',
  'original_title': 'Dog Day Afternoon',
  'overview': "Based on the true story of would-be Brooklyn bank robbers John Wojtowicz and Salvatore Naturale. Sonny and Sal attempt a bank heist which quickly turns sour and escalates into a hostage situation and stand-off with the police. As Sonny's motives for the robbery are slowly revealed and things become more complicated, the heist turns into a media circus.",
  'poster_path': 'https://image.tmdb.org/t/p/w500/3mYy8sLQWMS8tVHcac6T8sWHA6D.jpg',
  'release_date': '1975-09-21',
  'release_year': 1975,
  'vote_average': 7.836,
  'vote_count': 2675,
  'runtime': 125,
  'genres': 'Crime, Drama, Thriller',
  'original_language': 'en',
  'popularity': 18.585,
  'budget': 1800000,
  'revenue': 46665856},
 'cluster': 9,
 'recommendations': [{'id': 9040,
   'title': 'Serpico',
   'original_title': 'Serpico',
   'overview': "Frank Serpico is an idealistic New York City cop who refuses to take bri